# xRAG compression demo

This tutorial uses the xRAG method to compress stories from the ROCStories dataset, and then retrieve them with RAG + context compression.

Run this notebook on a large GPU node.

## Load ROCStories

First compress and 'encode' stories.

The query uses the first sentence of a story and asks xRAG to recover what happened next.

In [1]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from llm_psychology.data.rocstories import load_rocstories

roc_limit = int(os.environ.get("XRAG_ROC_LIMIT", "128"))
stories = load_rocstories(limit=roc_limit, shuffle=True, seed=13)
documents = stories["text"].astype(str).tolist()

row = stories.iloc[1]
question = f"{row['sentence1']} What happened next?"
target = str(row["continuation"])

print(f"Loaded {len(stories)} ROCStories documents")
print("Question:", question)
print("Target continuation:", target)

Loaded 128 ROCStories documents
Question: Yancy worked in a coal mine. What happened next?
Target continuation: One morning the mine collapsed. Yancy gathered his fellow miners together. They started digging a hole. With Yancy's lead, they were free by the end of the day.


## Load xRAG components

In [2]:
import os
import torch

from llm_psychology.xrag import (
    DEFAULT_XRAG_LLM,
    DEFAULT_XRAG_RETRIEVER,
    XRAG,
    XRAGConfig,
)


def optional_int(name):
    value = os.environ.get(name)
    return None if value in (None, "") else int(value)

config = XRAGConfig(
    llm_name=os.environ.get("XRAG_LLM", DEFAULT_XRAG_LLM),
    retriever_name=os.environ.get("XRAG_RETRIEVER", DEFAULT_XRAG_RETRIEVER),
    retriever_batch_size=int(os.environ.get("XRAG_BATCH_SIZE", "16")),
    retriever_max_length=int(os.environ.get("XRAG_RETRIEVER_MAX_LENGTH", "256")),
    retriever_query_clip_chars=optional_int("XRAG_QUERY_CLIP_CHARS"),
    docs_per_datastore=optional_int("XRAG_DOCS_PER_DATASTORE"),
)

device = os.environ.get("XRAG_DEVICE")
xrag = XRAG(config, device=device)
print("Device:", xrag.device)
print("LLM:", config.llm_name)
print("Retriever:", config.retriever_name)

xrag.load()
print("xRAG token id:", xrag.xrag_token_id)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

/nfs/nhome/live/espens/.pyenv/versions/3.11.5/envs/my_venv_3115/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Device: cuda
LLM: Hannibal046/xrag-7b
Retriever: Salesforce/SFR-Embedding-Mistral


Loading checkpoint shards: 100%|██████████████████| 3/3 [00:05<00:00,  1.67s/it]


xRAG token id: 32001
NVIDIA A100-SXM4-40GB


## Build xRAG datastore

First we get the SFR embedding for each ROCStories document, then the xRAG model projector maps the raw document embedding into xRAG space, and both raw and projected embeddings are cached. 

In [3]:
datastore = xrag.prepare_datastore(documents)

print("Documents:", len(datastore.documents))
print("Raw SFR document embeddings:", tuple(datastore.raw_doc_embeddings.shape))
print("Projected xRAG document embeddings:", tuple(datastore.xrag_doc_embeddings.shape))

Documents: 128
Raw SFR document embeddings: (128, 4096)
Projected xRAG document embeddings: (128, 4096)


## Retrieve and inspect the compressed document:

In [4]:
from llm_psychology.xrag import token_count, xrag_prompt_token_count

prompt, query_embedding = xrag.prepare_prompt(question)
retrieval = xrag.nearest_doc_embedding(query_embedding, datastore)
selected_doc = datastore.documents[retrieval.selected_index]

selected_doc_tokens = token_count(xrag.tokenizer, selected_doc)
prompt_tokens = xrag_prompt_token_count(xrag.tokenizer, prompt)

print("Selected document index:", retrieval.selected_index)
print("Nearest projected-space distance:", float(retrieval.distances[retrieval.selected_index]))
print("Selected document:", selected_doc)
print()
print(f"Selected document tokens compressed into one <xRAG> token: {selected_doc_tokens}")
print(f"Prompt tokens with <xRAG>: {prompt_tokens}")

Selected document index: 1
Nearest projected-space distance: 7481.73388671875
Selected document: Yancy worked in a coal mine. One morning the mine collapsed. Yancy gathered his fellow miners together. They started digging a hole. With Yancy's lead, they were free by the end of the day.

Selected document tokens compressed into one <xRAG> token: 46
Prompt tokens with <xRAG>: 46


## Generate with xRAG

The run_xrag call uses a prompt like this to get the 'recalled' story from the compressed 'memory' with RAG: 
```
<s>[INST] Refer to the background document and answer the question.

Background: <xRAG>

Question: {question} [/INST] The answer is:
```

The code replaces the `<xRAG>` string with the retrieved token embedding.

In [5]:
max_new_tokens = int(os.environ.get("XRAG_MAX_NEW_TOKENS", "64"))

text = xrag.run_xrag(prompt, retrieval.raw_embedding, max_new_tokens=max_new_tokens)
print("Target continuation:", target)
print()
print(text)

Target continuation: One morning the mine collapsed. Yancy gathered his fellow miners together. They started digging a hole. With Yancy's lead, they were free by the end of the day.

Yancy and his coworkers were trapped in the coal mine. They worked together to dig a tunnel to escape. They were eventually rescued.


Run this cell when you are finished with the notebook session:

In [6]:
xrag.release()
print("Released xRAG components")

Released xRAG components
